# Modelo ESG — Sem Data Leakage

## Por que esse notebook existe?

No projeto original, as features usadas (`environment_grade`, `social_grade`, `governance_grade`) eram **derivadas matematicamente do target** (`total_level`). O modelo aprendia uma equação trivial, não um padrão real — por isso chegava a 100% em todas as métricas.

Esse notebook usa **apenas as 3 features que são genuinamente independentes do target**:
- `industry` — setor da empresa
- `exchange` — bolsa onde é listada (NYSE ou NASDAQ)
- `currency` — moeda

O modelo vai ter desempenho menor (esperado ~60-70%), mas vai ser **honesto** — o que ele aprender vai ser um padrão real, não uma trapaça.

## 1. Imports e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Imports OK!')

In [ ]:
# Caminho do parquet raw (bronze)
BRONZE_PATH = '../data/bronze/esg_reporting_bronze.parquet'

df_raw = pd.read_parquet(BRONZE_PATH)

print(f'Shape: {df_raw.shape}')
df_raw.head(3)

## 2. Entendendo o target

O target é `total_level`. Vamos verificar a distribuição — datasets desbalanceados afetam o treinamento.

In [ ]:
print('Distribuição do total_level:')
print(df_raw['total_level'].value_counts())
print(f"\nProporção High: {df_raw['total_level'].value_counts(normalize=True)['High']:.1%}")

df_raw['total_level'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], rot=0)
plt.title('Distribuição do Target (total_level)')
plt.ylabel('Quantidade')
plt.show()

# O dataset está levemente desbalanceado (~62% High, ~38% Medium)
# Não é crítico, mas vamos usar métricas weighted para levar isso em conta

## 3. Explorando as 3 features legítimas

Antes de treinar, vale entender se essas features têm alguma relação com o target. Se não houver nenhuma relação, o modelo não vai ter o que aprender.

In [ ]:
# Criamos o target binário aqui para análise
df_raw['target'] = (df_raw['total_level'] == 'High').astype(int)

# Taxa de High por indústria
industry_high_rate = (
    df_raw.groupby('industry')['target']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_high', 'count': 'qtd_empresas'})
    .sort_values('taxa_high', ascending=False)
)

# Filtra indústrias com ao menos 5 empresas (amostras pequenas distorcem)
industry_high_rate = industry_high_rate[industry_high_rate['qtd_empresas'] >= 5]

plt.figure(figsize=(12, 7))
plt.barh(industry_high_rate.index, industry_high_rate['taxa_high'], color='steelblue')
plt.axvline(0.5, color='red', linestyle='--', label='50% (linha base)')
plt.xlabel('Taxa de ESG High')
plt.title('Taxa de ESG High por Indústria (mín. 5 empresas)')
plt.legend()
plt.tight_layout()
plt.show()

print('\nHá variação entre indústrias? Sim — isso significa que o modelo pode aprender algo real.')

In [ ]:
# Taxa de High por exchange
print('Taxa de High por Exchange:')
print(df_raw.groupby('exchange')['target'].mean().round(3))

print('\nTaxa de High por Currency:')
print(df_raw.groupby('currency')['target'].agg(['mean','count']))

# Observação: exchange tem alguma diferença entre NYSE e NASDAQ
# Currency é quase toda USD (97%), então vai ter pouco poder preditivo

## 4. Preparação dos dados

### 4.1 Seleção de features

Usamos **somente** as features que não derivam do target:
- `industry`, `exchange`, `currency` → categóricas, precisam de encoding

### 4.2 Encoding

Usamos `LabelEncoder` para transformar texto em números. **Importante:** salvamos os encoders para poder transformar dados novos no futuro (sem isso, o modelo não funciona em produção).

In [ ]:
FEATURES = ['industry', 'exchange', 'currency']
TARGET = 'target'

df = df_raw[FEATURES + [TARGET]].copy()

# Preenche nulos em industry com 'Unknown' antes de encodar
df['industry'] = df['industry'].fillna('Unknown')

print(f'Dataset: {df.shape}')
print(f'Nulos restantes:\n{df.isnull().sum()}')
df.head()

In [ ]:
# Encoding
encoders = {}

for col in FEATURES:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le  # salva o encoder para uso futuro

print('Encoding concluído. Valores únicos por feature:')
for col in FEATURES:
    print(f'  {col}: {df[col].nunique()} categorias')

df.head()

In [ ]:
# Salvamos os encoders — isso é importante!
# Sem eles, não conseguimos transformar dados novos da mesma forma que treinamos
os.makedirs('artifacts', exist_ok=True)
joblib.dump(encoders, 'artifacts/encoders.pkl')
print('Encoders salvos em artifacts/encoders.pkl')

# Para carregar depois:
# encoders = joblib.load('artifacts/encoders.pkl')

## 5. Divisão Treino / Teste

Dividimos os dados **antes** de treinar qualquer modelo. O conjunto de teste (`X_test`, `y_test`) **não pode ser visto pelo modelo durante o treino** — ele existe só para avaliar o desempenho real no final.

Usamos `stratify=y` para manter a mesma proporção de High/Medium em treino e teste.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')
print(f'\nProporção High no treino: {y_train.mean():.1%}')
print(f'Proporção High no teste:  {y_test.mean():.1%}')

## 6. Configuração do MLflow

O MLflow registra os parâmetros e métricas de cada treinamento para você comparar os modelos depois.

Usamos caminho absoluto para o banco (`mlflow.db`) para evitar erros de "arquivo não encontrado" dependendo de onde o notebook é rodado.

In [ ]:
# Caminho absoluto para o mlflow.db na raiz do projeto
ROOT = os.path.abspath('..')
MLFLOW_DB = os.path.join(ROOT, 'mlflow.db')

mlflow.set_tracking_uri(f'sqlite:///{MLFLOW_DB}')
mlflow.set_experiment('esg-tres-features')

print(f'MLflow configurado. Banco em: {MLFLOW_DB}')
print('Experimento: esg-tres-features')
print('\nPara abrir a UI do MLflow, rode no terminal (na raiz do projeto):')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')

## 7. Função auxiliar de avaliação

Criamos uma função reutilizável que avalia o modelo no conjunto de teste e plota a matriz de confusão.

In [ ]:
def avaliar_modelo(modelo, X_test, y_test, nome_modelo):
    """Avalia o modelo no conjunto de teste e retorna as métricas."""
    y_pred = modelo.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')

    print(f'\n=== {nome_modelo} — Resultado no Teste ===')
    print(f'  Accuracy:  {acc:.3f}')
    print(f'  Precision: {prec:.3f}')
    print(f'  Recall:    {rec:.3f}')
    print(f'  F1:        {f1:.3f}')
    print(f'\nRelatório completo:')
    print(classification_report(y_test, y_pred, target_names=['Medium (0)', 'High (1)']))

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['Medium', 'High'],
        cmap='Blues'
    )
    plt.title(f'Matriz de Confusão — {nome_modelo}')
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Função de avaliação e StratifiedKFold definidos.')

## 8. Modelo 1 — Decision Tree

A Decision Tree é um bom modelo inicial: é interpretável (dá para ver as regras que ela aprendeu) e funciona bem com features categóricas já encodadas.

Usamos `GridSearchCV` com `StratifiedKFold` para encontrar os melhores hiperparâmetros via cross-validation — isso evita que o modelo seja otimizado só para um subset de dados.

In [ ]:
tree_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

tree_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=tree_params,
    scoring='f1_weighted',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

tree_grid.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {tree_grid.best_params_}')
print(f'Melhor F1 no CV (treino): {tree_grid.best_score_:.3f}')

In [ ]:
best_tree = tree_grid.best_estimator_
tree_metrics = avaliar_modelo(best_tree, X_test, y_test, 'Decision Tree')

In [ ]:
with mlflow.start_run(run_name='decision_tree_3features'):
    mlflow.log_params(tree_grid.best_params_)
    mlflow.log_metric('cv_f1_treino',   tree_grid.best_score_)
    mlflow.log_metric('test_accuracy',  tree_metrics['accuracy'])
    mlflow.log_metric('test_precision', tree_metrics['precision'])
    mlflow.log_metric('test_recall',    tree_metrics['recall'])
    mlflow.log_metric('test_f1',        tree_metrics['f1'])
    mlflow.sklearn.log_model(best_tree, 'tree_model')
    run_id_tree = mlflow.active_run().info.run_id

print(f'Decision Tree logada! Run ID: {run_id_tree}')

## 9. Modelo 2 — KNN

O KNN classifica um ponto com base nos seus `k` vizinhos mais próximos. Com poucas features categóricas, pode ser menos eficaz que a árvore, mas vale comparar.

In [ ]:
knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
}

knn_grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_params,
    scoring='f1_weighted',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

knn_grid.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {knn_grid.best_params_}')
print(f'Melhor F1 no CV (treino): {knn_grid.best_score_:.3f}')

In [ ]:
best_knn = knn_grid.best_estimator_
knn_metrics = avaliar_modelo(best_knn, X_test, y_test, 'KNN')

In [ ]:
with mlflow.start_run(run_name='knn_3features'):
    mlflow.log_params(knn_grid.best_params_)
    mlflow.log_metric('cv_f1_treino',   knn_grid.best_score_)
    mlflow.log_metric('test_accuracy',  knn_metrics['accuracy'])
    mlflow.log_metric('test_precision', knn_metrics['precision'])
    mlflow.log_metric('test_recall',    knn_metrics['recall'])
    mlflow.log_metric('test_f1',        knn_metrics['f1'])
    mlflow.sklearn.log_model(best_knn, 'knn_model')
    run_id_knn = mlflow.active_run().info.run_id

print(f'KNN logado! Run ID: {run_id_knn}')

## 10. Modelo 3 — XGBoost

O XGBoost é um modelo baseado em árvores de decisão combinadas (ensemble), geralmente mais poderoso que uma única árvore. Ele constrói várias árvores em sequência, onde cada nova tenta corrigir os erros da anterior.

Com apenas 3 features categóricas, pode não ter grande vantagem sobre a Decision Tree, mas o GridSearch vai encontrar a configuração mais adequada.

In [ ]:
xgb_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=xgb_params,
    scoring='f1_weighted',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {xgb_grid.best_params_}')
print(f'Melhor F1 no CV (treino): {xgb_grid.best_score_:.3f}')

In [ ]:
best_xgb = xgb_grid.best_estimator_
xgb_metrics = avaliar_modelo(best_xgb, X_test, y_test, 'XGBoost')

In [ ]:
with mlflow.start_run(run_name='xgboost_3features'):
    mlflow.log_params(xgb_grid.best_params_)
    mlflow.log_metric('cv_f1_treino',   xgb_grid.best_score_)
    mlflow.log_metric('test_accuracy',  xgb_metrics['accuracy'])
    mlflow.log_metric('test_precision', xgb_metrics['precision'])
    mlflow.log_metric('test_recall',    xgb_metrics['recall'])
    mlflow.log_metric('test_f1',        xgb_metrics['f1'])
    mlflow.sklearn.log_model(best_xgb, 'xgb_model')
    run_id_xgb = mlflow.active_run().info.run_id

print(f'XGBoost logado! Run ID: {run_id_xgb}')

## 11. Comparação dos 3 modelos

Comparamos os três modelos pelas métricas no conjunto de **teste**. A linha pontilhada é o baseline — se o modelo simplesmente chutasse sempre "High" (classe majoritária), acertaria ~62% das vezes.

In [ ]:
resultados = pd.DataFrame({
    'Decision Tree': tree_metrics,
    'KNN':           knn_metrics,
    'XGBoost':       xgb_metrics,
}).T.round(3)

print('=== Comparação no Conjunto de Teste ===')
print(resultados.to_string())

resultados.plot(
    kind='bar', figsize=(11, 5), ylim=(0, 1), rot=0,
    color=['steelblue', 'tomato', 'seagreen', 'gold']
)
plt.title('Comparação de Métricas no Teste — 3 Modelos')
plt.ylabel('Score')
plt.axhline(0.62, color='gray', linestyle='--', label='baseline (classe majoritária)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 12. Registrando o melhor modelo no MLflow Model Registry

O Model Registry é uma funcionalidade do MLflow onde você "nomeia" o melhor modelo. Assim é fácil carregar sempre a versão correta — sem precisar lembrar o run_id.

In [ ]:
# Escolhe o melhor modelo pelo F1 no teste
modelos = {
    'Decision Tree': (tree_metrics['f1'], run_id_tree, 'tree_model'),
    'KNN':           (knn_metrics['f1'],  run_id_knn,  'knn_model'),
    'XGBoost':       (xgb_metrics['f1'],  run_id_xgb,  'xgb_model'),
}

melhor_nome, (melhor_f1, melhor_run_id, melhor_artifact) = max(
    modelos.items(), key=lambda x: x[1][0]
)

print(f'Melhor modelo: {melhor_nome} (F1 = {melhor_f1:.3f})')

# Registra no Model Registry com um nome fixo
mlflow.register_model(
    model_uri=f'runs:/{melhor_run_id}/{melhor_artifact}',
    name='esg-classificador-3features'
)

print('\nModelo registrado como: esg-classificador-3features')
print('\nPara carregar qualquer vez no futuro:')
print('  model = mlflow.sklearn.load_model("models:/esg-classificador-3features/1")')

## 13. Como usar o modelo salvo

Exemplo de como carregar o modelo e os encoders para fazer predições em dados novos.

In [ ]:
# Carrega o modelo do MLflow
model_carregado = mlflow.sklearn.load_model('models:/esg-classificador-3features/1')

# Carrega os encoders
encoders_carregados = joblib.load('artifacts/encoders.pkl')

# Exemplo de empresa nova
nova_empresa = pd.DataFrame([{
    'industry': 'Technology',
    'exchange': 'NASDAQ NMS - GLOBAL MARKET',
    'currency': 'USD'
}])

# Aplica os mesmos encoders do treino
for col in FEATURES:
    nova_empresa[col] = encoders_carregados[col].transform(nova_empresa[col])

# Predição
pred = model_carregado.predict(nova_empresa)
prob = model_carregado.predict_proba(nova_empresa)

resultado = 'High' if pred[0] == 1 else 'Medium'
print(f'Predição: {resultado}')
print(f'Probabilidade Medium: {prob[0][0]:.1%}')
print(f'Probabilidade High:   {prob[0][1]:.1%}')

## 14. Conclusão

**O que esse notebook fez de diferente do projeto original:**

1. **Sem data leakage** — usou apenas `industry`, `exchange` e `currency`, que são características reais da empresa e não derivadas do target.

2. **Avaliou no conjunto de teste** — as métricas reportadas são do `X_test`, que o modelo nunca viu durante o treino.

3. **Salvou os encoders** — com `joblib`, para que seja possível transformar dados novos da mesma forma.

4. **Registrou o melhor modelo no MLflow Model Registry** — comparou Decision Tree, KNN e XGBoost e registrou o vencedor pelo F1 no teste.

**Próximos passos sugeridos:**
- Enriquecer o dataset com features externas (ex: tamanho da empresa, região, nível de regulação do setor) para melhorar a acurácia sem introduzir leakage.
- Explorar um modelo de Regressão Logística, que é mais simples e costuma ser bom baseline para features categóricas.